## Universal Run Checklist

Primary path (recommended every run):
1. **Set protocol + options** in the experiment parameter section (`bender.test_type`, save path, `use_sono`, calibration options).
2. **Validate setup (optional):** `bender.validate_dispatch_setup()`.
3. **Run universal pre-run previews** (table + plots).
4. **Run universal dispatcher:** `bender.run_experiment(test_type=bender.test_type)`.
5. **Calibration only:** run inertial calibration save cell to create standalone calibration H5.
6. **Run PRIMARY exporter:** `Save ALL data (PRIMARY export path)`.
7. **Run universal QC plot** and save figure.
8. **Write post-trial notes** (saved to H5 key `post-trial notes`).


In [32]:
# Universal notebook controls

# Optional pre-run validation for GUI-style dispatcher setup.
# Call this after parameter cell edits:
# print(bender.validate_dispatch_setup())


DEBUG: Loading bender_functions.py from: c:\Users\jimen\Desktop\BenderCode\Bender_PC2026\bender_functions.py


# Import Dependencies

In [33]:
#region
#  1. Data Management (Always first)
%load_ext autoreload
%autoreload 2

# 2. Objects you are actually manipulating in cells
import numpy as np
import h5py
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 3. The Bridge to your script
from bender_functions import Bender
#endregion

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Initialize + Morphometrics + Experiment Parameters (single setup chunk)

In [34]:
# --- Single setup chunk: initialize + morphometrics + experiment parameters ---
bender = Bender(config_module_name="jimenez_bender_config_A")
summary = bender.summary()

# ---------------- File / output ----------------
data_folder = r"c:\Users\jimen\Desktop\BenderData\RandomTest"
base_name = "2026-04-01_Polyurethane40A"
bender.outputfile = bender.increment_file_name(f"{data_folder}\\{base_name}.h5")
bender.outputfig = bender.outputfile.replace(".h5", "_qc.png")

# ---------------- Protocol selection + required inputs ----------------
# Choose one: dynamic | frequency_sweep | frequency_step | curvature_step | step_change | isometric | isovelocity | calibration
bender.test_type = "isovelocity"

# Required for dynamic/frequency_*/curvature_step/step_change:
# - all_freqs, all_amps, all_amps_mode, duration, cycles_per_step
# Required for isometric:
# - isometric_initial, isometric_final, isometric_num_steps, isometric_mode
# Required for isovelocity:
# - isovelocity_min_vel, isovelocity_max_vel, isovelocity_starting_strain, isovelocity_num_steps
# Required for calibration:
# - calibration_base_test_type (dynamic/frequency_sweep/frequency_step/curvature_step/step_change)

# Motion-series defaults (dynamic/frequency_* family)
all_freqs = [1, 5]
all_amps = [4, 6.5]
all_amps_mode = "strain"   # curvature | strain_pct | strain | angle
cycles_per_step = 5
n_end_cycles = 2
duration = 15
stim_cycles_in_step = np.array([2, 3])
randomize = False

# Isometric inputs
isometric_initial = 0.01
isometric_final = 0.04
isometric_num_steps = 5
isometric_mode = "strain"
isometric_randomize = False
isometric_random_seed = None
isometric_stim_params = {
    "ramp_duration_s": 2.0,
    "hold_duration_s": 5.0,
    "settle_before_stim_s": 0.5,
    "stim_duration_s": 0.2,
}

# Isovelocity inputs
isovelocity_min_vel = 2.0
isovelocity_max_vel = 20.0
isovelocity_starting_strain = 0.02
isovelocity_starting_strain_mode = "strain"
isovelocity_num_steps = 8
isovelocity_randomize = False
isovelocity_random_seed = None
isovelocity_iso_duration_s = 0.2
isovelocity_pre_hold_s = 0.3
isovelocity_stim_params = {
    "pre_iso_stim_duration_s": 0.05,
    "settle_before_stim_s": 0.02,
    "stim_duration_s": 0.10,
}

# Calibration-only input
calibration_base_test_type = "dynamic"

if bender.test_type in ("dynamic", "frequency_sweep", "frequency_step", "curvature_step", "step_change"):
    protocol_params = {
        "type": bender.test_type,
        "randomize": randomize,
        "cycles_per_step": cycles_per_step,
        "n_end_cycles": n_end_cycles,
        "duration": duration,
        "stim_cycles_in_step": stim_cycles_in_step,
    }
elif bender.test_type == "isometric":
    protocol_params = {
        "type": "isometric",
        "initial": isometric_initial,
        "final": isometric_final,
        "num_steps": isometric_num_steps,
        "mode": isometric_mode,
        "randomize": isometric_randomize,
        "random_seed": isometric_random_seed,
        "stim_params": isometric_stim_params,
    }
elif bender.test_type == "isovelocity":
    protocol_params = {
        "type": "isovelocity",
        "min_vel": isovelocity_min_vel,
        "max_vel": isovelocity_max_vel,
        "starting_strain": isovelocity_starting_strain,
        "starting_strain_mode": isovelocity_starting_strain_mode,
        "num_steps": isovelocity_num_steps,
        "randomize": isovelocity_randomize,
        "random_seed": isovelocity_random_seed,
        "iso_duration_s": isovelocity_iso_duration_s,
        "pre_hold_s": isovelocity_pre_hold_s,
        "stim_params": isovelocity_stim_params,
    }
elif bender.test_type == "calibration":
    protocol_params = {
        "type": "calibration",
        "base_test_type": calibration_base_test_type,
    }
else:
    raise ValueError(f"Unsupported test_type: {bender.test_type}")

# ---------------- Morphometrics ----------------
fishcode = "40A polyurethane rod"
segment = "NA"
fishmass = "NA"      # grams
fishlen_TL = 304.8    # mm
fishlen_SL = 304.8    # mm
xsec_width = 25.4     # mm
xsec_height = 25.4    # mm
dbend = 180           # mm
dclamp = 52           # mm
dvert = 47            # mm
dhoriz = 0            # mm

# ---------------- Feature toggles ----------------
use_sono = True
primary_bending_axis = "zTorque"   # xTorque | yTorque | zTorque (or x/y/z)

# ---------------- Specimen profile inertial model inputs ----------------
# Preferred: any number of motor-axis stations with width/height.
# Tiny helper path: proximal/distal + optional middle.
use_frustum_inertial_model = True
use_theoretical_inertial_correction = False  # apply MOI*alpha when no calibration profile

proximal_height_mm = xsec_height
proximal_width_mm = xsec_width
middle_height_mm = None     # set numeric value to include middle station
middle_width_mm = None      # set numeric value to include middle station
middle_position = 0.5

distal_height_mm = xsec_height
distal_width_mm = xsec_width

specimen_profile_stations = bender.make_profile_stations(
    proximal_height_mm=proximal_height_mm,
    proximal_width_mm=proximal_width_mm,
    distal_height_mm=distal_height_mm,
    distal_width_mm=distal_width_mm,
    middle_height_mm=middle_height_mm,
    middle_width_mm=middle_width_mm,
    middle_position=middle_position,
)
# Advanced alternative: pass any number of custom stations directly as a list of dicts.
# specimen_profile_stations = [
#     {"label": "proximal", "position": 0.0, "height_mm": 30, "width_mm": 28},
#     {"label": "middle",   "position": 0.4, "height_mm": 24, "width_mm": 22},
#     {"label": "distal",   "position": 1.0, "height_mm": 18, "width_mm": 16},
# ]

# Specimen profile length = total length - dbend - half clamp spacing
specimen_profile_length_mm = fishlen_TL - dbend - (0.5 * dclamp)
if specimen_profile_length_mm <= 0:
    raise ValueError(
        f"Computed specimen_profile_length_mm must be > 0; got {specimen_profile_length_mm}. "
        "Check fishlen_TL, dbend, and dclamp values."
    )
specimen_profile_density_g_per_mm3 = 0.001
specimen_profile_clamp_offset_mm = 20.0
specimen_profile_num_samples = 120

# ---------------- Sweep / ramp knobs ----------------
amplitude_frequency_exponent = 0
velocity_exponent = 1.0

# ---------------- Stimulation inputs ----------------
# Minimum by test_type:
# - dynamic/frequency_*: set is_stim + all_stimduties + all_stimphases (+ pulse/voltage fields)
# - isometric: set is_stim + stim_pulse_rate + S1volts/S2volts (timing is handled by protocol params)
# - isovelocity: set is_stim + stim_pulse_rate + S1volts/S2volts (pre/iso timing via protocol params)
# - calibration: typically is_stim=False
is_stim = True
all_stimduties = [0]
all_stimphases = [0]
stim_pulse_rate = 75
S1volts = 10
S2volts = 10
S1pulsedur = 2
S2pulsedur = 2

# ---------------- Inertial calibration linkage ----------------
use_inertial_calibration = False
inertial_calibration_file = ""

# ---------------- Single metadata update call ----------------
bender.update_metadata(
    # File / protocol
    test_type=bender.test_type,
    protocol_params=protocol_params,
    # Morphometrics
    fishcode=fishcode,
    segment=segment,
    fishmass=fishmass,
    fishlen_TL=fishlen_TL,
    fishlen_SL=fishlen_SL,
    xsec_width=xsec_width,
    xsec_height=xsec_height,
    dbend=dbend,
    dclamp=dclamp,
    dvert=dvert,
    dhoriz=dhoriz,
    # Motion / amplitude
    all_freqs=all_freqs,
    all_amps=all_amps,
    all_amps_mode=all_amps_mode,
    randomize=randomize,
    cycles_per_step=cycles_per_step,
    n_end_cycles=n_end_cycles,
    stim_cycles_in_step=stim_cycles_in_step,
    duration=duration,
    # Toggles / calibration
    use_sono=use_sono,
    primary_bending_axis=primary_bending_axis,
    use_inertial_calibration=use_inertial_calibration,
    inertial_calibration_file=inertial_calibration_file,
    # Specimen profile inertial model
    use_frustum_inertial_model=use_frustum_inertial_model,
    use_theoretical_inertial_correction=use_theoretical_inertial_correction,
    specimen_profile_stations=specimen_profile_stations,
    specimen_profile_length_mm=specimen_profile_length_mm,
    specimen_profile_density_g_per_mm3=specimen_profile_density_g_per_mm3,
    specimen_profile_clamp_offset_mm=specimen_profile_clamp_offset_mm,
    specimen_profile_num_samples=specimen_profile_num_samples,
    # Sweep knobs
    amplitude_frequency_exponent=amplitude_frequency_exponent,
    velocity_exponent=velocity_exponent,
    # Stimulation
    is_stim=is_stim,
    all_stimduties=all_stimduties,
    all_stimphases=all_stimphases,
    stim_pulse_rate=stim_pulse_rate,
    S1volts=S1volts,
    S2volts=S2volts,
    S1pulsedur=S1pulsedur,
    S2pulsedur=S2pulsedur,
)
print(f"DEBUG: After update_metadata, bender.test_type is {bender.test_type!r}")

# ---------------- Pre-experiment check ----------------
precheck = pd.DataFrame([
    {"Parameter": "test_type", "Value": bender.test_type},
    {"Parameter": "protocol_params", "Value": protocol_params},
    {"Parameter": "all_amps_mode", "Value": all_amps_mode},
    {"Parameter": "all_amps", "Value": all_amps},
    {"Parameter": "all_curves (1/m)", "Value": getattr(bender, "all_curves", None)},
    {"Parameter": "all_freqs (Hz)", "Value": all_freqs},
    {"Parameter": "use_sono", "Value": use_sono},
    {"Parameter": "primary_bending_axis", "Value": primary_bending_axis},
    {"Parameter": "use_inertial_calibration", "Value": use_inertial_calibration},
    {"Parameter": "inertial_calibration_file", "Value": inertial_calibration_file},
    {"Parameter": "use_frustum_inertial_model", "Value": use_frustum_inertial_model},
    {"Parameter": "use_theoretical_inertial_correction", "Value": use_theoretical_inertial_correction},
    {"Parameter": "specimen_profile_length_mm", "Value": specimen_profile_length_mm},
    {"Parameter": "length formula", "Value": "fishlen_TL - dbend - 0.5*dclamp"},
    {"Parameter": "specimen_profile_density_g_per_mm3", "Value": specimen_profile_density_g_per_mm3},
    {"Parameter": "proximal_height_mm", "Value": proximal_height_mm},
    {"Parameter": "proximal_width_mm", "Value": proximal_width_mm},
    {"Parameter": "middle_height_mm", "Value": middle_height_mm},
    {"Parameter": "middle_width_mm", "Value": middle_width_mm},
    {"Parameter": "distal_height_mm", "Value": distal_height_mm},
    {"Parameter": "distal_width_mm", "Value": distal_width_mm},
    {"Parameter": "specimen_profile_stations", "Value": specimen_profile_stations},
    {"Parameter": "i_total_system (g*mm^2)", "Value": getattr(bender, "i_total_system", None)},
    {"Parameter": "fishcode", "Value": fishcode},
    {"Parameter": "xsec_width (mm)", "Value": xsec_width},
    {"Parameter": "xsec_height (mm)", "Value": xsec_height},
    {"Parameter": "dclamp (mm)", "Value": dclamp},
    {"Parameter": "outputfile (.h5)", "Value": bender.outputfile},
    {"Parameter": "outputfig (QC)", "Value": bender.outputfig},
])
pd.set_option("display.max_colwidth", None)
display(precheck)


Bender initialized using: jimenez_bender_config_A.py
              BENDER SYSTEM SUMMARY               
Config:      jimenez_bender_config_A.py
Device:      Dev1
Motor Port:  port0
Direction:   POSITIVE = LEFT
--------------------------------------------------
Cal File:    FT56491.cal
Sample Rate: 1000.0 Hz
Ramp:        0.25 s
  Stored: test_type = isovelocity
  Stored: fishcode = 40A polyurethane rod
  Stored: segment = NA
  Stored: fishmass = NA
  Stored: fishlen_TL = 304.8
  Stored: fishlen_SL = 304.8
  Stored: xsec_width = 25.4
  Stored: xsec_height = 25.4
  Stored: dbend = 180
  Stored: dclamp = 52
  Stored: dvert = 47
  Stored: dhoriz = 0
  Stored: all_freqs = [1, 5]
  Stored: all_amps = [4, 6.5]
  Stored: all_amps_mode = strain
  Stored: randomize = False
  Stored: cycles_per_step = 5
  Stored: n_end_cycles = 2
  Stored: stim_cycles_in_step = [2 3]
  Stored: duration = 15
  Stored: use_sono = True
  Stored: primary_bending_axis = zTorque
  Stored: use_inertial_calibration = Fals

,Parameter,Value
0,test_type,isovelocity
1,protocol_params,"{'type': 'isovelocity', 'min_vel': 2.0, 'max_vel': 20.0, 'starting_strain': 0.02, 'starting_strain_mode': 'strain', 'num_steps': 8, 'randomize': False, 'random_seed': None, 'iso_duration_s': 0.2, 'pre_hold_s': 0.3, 'stim_params': {'pre_iso_stim_duration_s': 0.05, 'settle_before_stim_s': 0.02, 'stim_duration_s': 0.1}}"
2,all_amps_mode,strain
3,all_amps,"[4, 6.5]"
4,all_curves (1/m),"[314.96062992125985, 511.8110236220473]"
5,all_freqs (Hz),"[1, 5]"
6,use_sono,True
7,primary_bending_axis,zTorque
8,use_inertial_calibration,False
9,inertial_calibration_file,


## Universal pre-run preview (table + plot)

This preview is DAQ-free and works for all `bender.test_type` values. Use it to verify sequence order and commanded angle timeline before pressing run.

For **isometric** / **isovelocity**, the plan table includes `recruitment`, angle previews follow the same left/right motor sign as the live protocol, and the separate **strain + stim preview** cell labels geometric strain (ε = κ·w/2) with the recruited side / bilateral mode.

In [35]:
# Universal DAQ-free preview for all dispatcher test types.
test_type_preview = str(getattr(bender, "test_type", "dynamic"))
preview_rows = []
preview_t = np.array([], dtype=float)
preview_angle = np.array([], dtype=float)
preview_vel = np.array([], dtype=float)

def _motion_preview(tt):
    # Reuse existing motion generators (no hardware run).
    bender.organize_cycles(
        all_curves=bender.all_curves,
        all_freqs=bender.all_freqs,
        randomize=bender.randomize,
        cycles_per_step=bender.cycles_per_step,
        n_end_cycles=bender.n_end_cycles,
        dclamp=bender.dclamp,
        xsec_width=bender.xsec_width,
        stim_cycles_in_step=bender.stim_cycles_in_step,
        all_stimduties=bender.all_stimduties,
        all_stimphases=bender.all_stimphases,
        stim_pulse_rate=bender.stim_pulse_rate,
    )
    if tt in ["dynamic", "static"]:
        angle, anglevel, _, t = bender.make_cycles_dynamic(
            bender.period_by_cycle, bender.freq_by_cycle, bender.amp_by_cycle
        )
    elif tt == "frequency_sweep":
        angle, anglevel, _, _, t = bender.make_cycles_frequency_sweep(
            bender.all_freqs, bender.all_curves, bender.amplitude_frequency_exponent, bender.duration, bender.waitbefore
        )
    elif tt == "frequency_step":
        angle, anglevel, _, _, t = bender.make_cycles_frequency_step(
            bender.all_freqs, bender.all_curves, bender.duration, bender.waitbefore
        )
    elif tt == "curvature_step":
        angle, anglevel, _, _, t = bender.make_cycles_curvature_step(
            bender.all_freqs, bender.all_curves, bender.duration, bender.waitbefore
        )
    elif tt == "step_change":
        angle, anglevel, _, t, _ = bender.make_cycles_step_change(
            bender.step_change_frequencies,
            bender.step_change_curves,
            bender.step_change_cycles_per_step,
            dclamp=getattr(bender, "dclamp", None),
            amp_step_vel=getattr(bender, "step_change_amp_step_vel", None),
        )
    else:
        raise ValueError(f"Unsupported motion preview type: {tt}")
    return np.asarray(t), np.asarray(angle), np.asarray(anglevel)

if test_type_preview == "calibration":
    base = str(getattr(bender, "calibration_base_test_type", "dynamic"))
    tt = base
else:
    tt = test_type_preview

if tt in ["dynamic", "static", "frequency_sweep", "frequency_step", "curvature_step", "step_change"]:
    preview_t, preview_angle, preview_vel = _motion_preview(tt)
    preview_rows.append({
        "trial_index": 0,
        "cycle_index": 0,
        "test_type": test_type_preview,
        "motion_type": tt,
        "n_samples": int(preview_t.size),
        "duration_s": float(preview_t[-1] - preview_t[0]) if preview_t.size > 1 else 0.0,
        "angle_min_deg": float(np.nanmin(preview_angle)) if preview_angle.size else np.nan,
        "angle_max_deg": float(np.nanmax(preview_angle)) if preview_angle.size else np.nan,
    })

elif tt == "isometric":
    bender._normalize_dispatch_aliases()
    initial = float(getattr(bender, "isometric_initial"))
    final = float(getattr(bender, "isometric_final"))
    num_steps = int(getattr(bender, "isometric_num_steps"))
    mode = str(getattr(bender, "isometric_mode", "strain"))
    randomize = bool(getattr(bender, "isometric_randomize", False))
    random_seed = getattr(bender, "isometric_random_seed", None)
    ramp_s = float(getattr(bender, "isometric_stim_params", {}).get("ramp_duration_s", 2.0))
    hold_s = float(getattr(bender, "isometric_stim_params", {}).get("hold_duration_s", 5.0))

    vals = np.linspace(initial, final, num_steps)
    seq = np.arange(vals.size)
    if randomize and vals.size > 1:
        rng = np.random.default_rng(None if random_seed is None else int(random_seed))
        rng.shuffle(seq)
        vals = vals[seq]

    kappa = bender.convert_to_curvature(vals, mode)
    targets_deg = np.rad2deg(kappa * (float(bender.dclamp) / 1000.0))
    rec_prev = bender._normalize_recruitment(getattr(bender, "recruitment", "bilateral_simultaneous"))
    uidx_prev = bender.unilateral_posture_lateral_index(rec_prev)
    if uidx_prev is not None:
        targets_deg = targets_deg * bender.motor_command_sign_for_bend_toward_index(uidx_prev)

    t_offset = 0.0
    for i, target in enumerate(targets_deg):
        prev = float(targets_deg[i - 1]) if i > 0 else float(targets_deg[0])
        t_i, a_i, w_i = bender._timeline_ramp_hold(prev, float(target), ramp_s, hold_s, float(bender.daq_ai_sample_rate_hz))
        preview_t = np.concatenate([preview_t, t_i + t_offset])
        preview_angle = np.concatenate([preview_angle, a_i])
        preview_vel = np.concatenate([preview_vel, w_i])
        t_offset = preview_t[-1] if preview_t.size > 0 else t_offset
        preview_rows.append({
            "trial_index": i,
            "cycle_index": i,
            "test_type": test_type_preview,
            "motion_type": "isometric",
            "recruitment": rec_prev,
            "unilateral_posture_lateral_index": uidx_prev,
            "target_native": float(vals[i]),
            "target_deg": float(target),
            "sequence_index": int(seq[i]),
        })

elif tt == "isovelocity":
    bender._normalize_dispatch_aliases()
    min_vel = float(getattr(bender, "isovelocity_min_vel"))
    max_vel = float(getattr(bender, "isovelocity_max_vel"))
    num_steps = int(getattr(bender, "isovelocity_num_steps"))
    starting_strain = float(getattr(bender, "isovelocity_starting_strain"))
    starting_mode = str(getattr(bender, "isovelocity_starting_strain_mode", "strain"))
    randomize = bool(getattr(bender, "isovelocity_randomize", False))
    random_seed = getattr(bender, "isovelocity_random_seed", None)
    iso_s = float(getattr(bender, "isovelocity_iso_duration_s", 0.2))
    pre_s = float(getattr(bender, "isovelocity_pre_hold_s", 0.3))

    k0 = bender.convert_to_curvature(starting_strain, starting_mode)
    theta0 = float(np.rad2deg(float(np.asarray(k0).reshape(-1)[0]) * (float(bender.dclamp) / 1000.0)))
    rec_prev = bender._normalize_recruitment(getattr(bender, "recruitment", "bilateral_simultaneous"))
    uidx_prev = bender.unilateral_posture_lateral_index(rec_prev)
    if uidx_prev is not None:
        theta0 = theta0 * bender.motor_command_sign_for_bend_toward_index(uidx_prev)

    vels = np.linspace(min_vel, max_vel, num_steps)
    seq = np.arange(vels.size)
    if randomize and vels.size > 1:
        rng = np.random.default_rng(None if random_seed is None else int(random_seed))
        rng.shuffle(seq)
        vels = vels[seq]

    t_offset = 0.0
    for i, v in enumerate(vels):
        v_use = float(v)
        if uidx_prev is not None:
            v_use = abs(v_use) * bender.motor_command_sign_for_bend_toward_index(uidx_prev)
        t_i, a_i, w_i, _ = bender._timeline_prehold_isovelocity(theta0, v_use, pre_s, iso_s, float(bender.daq_ai_sample_rate_hz))
        preview_t = np.concatenate([preview_t, t_i + t_offset])
        preview_angle = np.concatenate([preview_angle, a_i])
        preview_vel = np.concatenate([preview_vel, w_i])
        t_offset = preview_t[-1] if preview_t.size > 0 else t_offset
        preview_rows.append({
            "trial_index": i,
            "cycle_index": i,
            "test_type": test_type_preview,
            "motion_type": "isovelocity",
            "recruitment": rec_prev,
            "unilateral_posture_lateral_index": uidx_prev,
            "velocity_deg_s": float(v_use),
            "theta_start_deg": float(theta0),
            "sequence_index": int(seq[i]),
        })

else:
    raise ValueError(f"Unsupported test_type preview: {test_type_preview}")

preview_plan_df = pd.DataFrame(preview_rows)
display(preview_plan_df)

fig_prev = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
if preview_t.size and preview_angle.size == preview_t.size:
    fig_prev.add_trace(go.Scatter(x=preview_t, y=preview_angle, mode="lines", name="angle_cmd"), row=1, col=1)
if preview_t.size and preview_vel.size == preview_t.size:
    fig_prev.add_trace(go.Scatter(x=preview_t, y=preview_vel, mode="lines", name="anglevel_cmd", line=dict(color="orange")), row=2, col=1)
fig_prev.update_yaxes(title_text="Angle (deg)", row=1, col=1)
fig_prev.update_yaxes(title_text="Velocity (deg/s)", row=2, col=1)
fig_prev.update_xaxes(title_text="Preview time (s)", row=2, col=1)
_prev_title = f"Pre-run preview: {test_type_preview}"
if test_type_preview in ("isometric", "isovelocity"):
    _prev_title = _prev_title + " | " + bender.strain_yaxis_title_pct()
fig_prev.update_layout(height=700, width=1100, title_text=_prev_title)
try:
    fig_prev.show(renderer="notebook_connected")
except Exception:
    fig_prev.show()
display(fig_prev)


,trial_index,cycle_index,test_type,motion_type,velocity_deg_s,theta_start_deg,sequence_index
0,0,0,isovelocity,isovelocity,2.000000,4.691938,0
1,1,1,isovelocity,isovelocity,4.571429,4.691938,1
2,2,2,isovelocity,isovelocity,7.142857,4.691938,2
3,3,3,isovelocity,isovelocity,9.714286,4.691938,3
4,4,4,isovelocity,isovelocity,12.285714,4.691938,4
5,5,5,isovelocity,isovelocity,14.857143,4.691938,5
6,6,6,isovelocity,isovelocity,17.428571,4.691938,6
7,7,7,isovelocity,isovelocity,20.000000,4.691938,7


## Pre-run stimulus + strain preview (universal)

Strain preview uses commanded motor angle and geometry:
- curvature `kappa = theta_rad / segment_length_m`
- strain fraction `epsilon = kappa * half_width_m`
- strain percent `epsilon_pct = 100 * epsilon`

where `segment_length_m = dclamp_mm / 1000` and `half_width_m = xsec_width_mm / 2000`.

In [36]:
# Requires the universal pre-run preview cell to be executed first.
if "preview_t" not in globals() or "preview_angle" not in globals():
    raise RuntimeError("Run the 'Universal pre-run preview (table + plot)' cell first.")

preview_t = np.asarray(preview_t, dtype=float)
preview_angle = np.asarray(preview_angle, dtype=float)

# --- Strain from geometry ---
dclamp_m = float(getattr(bender, "dclamp", np.nan)) / 1000.0
half_width_m = float(getattr(bender, "xsec_width", np.nan)) / 2000.0
if not np.isfinite(dclamp_m) or dclamp_m <= 0:
    raise ValueError("dclamp must be positive to compute strain preview.")
if not np.isfinite(half_width_m) or half_width_m <= 0:
    raise ValueError("xsec_width must be positive to compute strain preview.")

theta_rad = np.deg2rad(preview_angle)
kappa_1pm = theta_rad / dclamp_m
epsilon = kappa_1pm * half_width_m
strain_pct_preview = 100.0 * epsilon * bender.strain_display_sign()

# --- Stimulus preview (approximate but protocol-aware) ---
is_stim = bool(getattr(bender, "is_stim", False))
stim_rate = float(getattr(bender, "stim_pulse_rate", 75.0))
s1v = float(getattr(bender, "S1volts", 0.0))
s2v = float(getattr(bender, "S2volts", 0.0))

s1_prev = np.zeros_like(preview_t)
s2_prev = np.zeros_like(preview_t)
if is_stim and preview_t.size > 0:
    pulse = (np.mod(preview_t * stim_rate, 1.0) <= 0.5).astype(float)
    active = np.zeros_like(preview_t, dtype=bool)

    tt = str(getattr(bender, "test_type", "dynamic"))
    if tt == "calibration":
        tt = str(getattr(bender, "calibration_base_test_type", "dynamic"))

    if tt == "isometric":
        sp = dict(getattr(bender, "isometric_stim_params", {}) or {})
        ramp_s = float(sp.get("ramp_duration_s", 2.0))
        hold_s = float(sp.get("hold_duration_s", 5.0))
        settle_s = float(sp.get("settle_before_stim_s", 0.5))
        stim_dur = sp.get("stim_duration_s", None)
        n_trials = int(getattr(bender, "isometric_num_steps", len(preview_plan_df) if "preview_plan_df" in globals() else 1))
        trial_len = ramp_s + hold_s
        for i in range(max(1, n_trials)):
            t0 = i * trial_len + ramp_s + settle_s
            t1 = i * trial_len + (ramp_s + hold_s if stim_dur is None else min(ramp_s + hold_s, ramp_s + settle_s + float(stim_dur)))
            active |= (preview_t >= t0) & (preview_t < t1)

    elif tt == "isovelocity":
        sp = dict(getattr(bender, "isovelocity_stim_params", {}) or {})
        pre_hold_s = float(getattr(bender, "isovelocity_pre_hold_s", 0.3))
        iso_s = float(getattr(bender, "isovelocity_iso_duration_s", 0.2))
        settle_s = float(sp.get("settle_before_stim_s", 0.02))
        pre_iso_stim = float(sp.get("pre_iso_stim_duration_s", 0.0))
        stim_dur = sp.get("stim_duration_s", None)
        n_trials = int(getattr(bender, "isovelocity_num_steps", len(preview_plan_df) if "preview_plan_df" in globals() else 1))
        trial_len = pre_hold_s + iso_s
        for i in range(max(1, n_trials)):
            t_iso0 = i * trial_len + pre_hold_s
            if pre_iso_stim > 0:
                active |= (preview_t >= max(i * trial_len, t_iso0 - pre_iso_stim)) & (preview_t < t_iso0)
            t0 = t_iso0 + settle_s
            t1 = t_iso0 + (iso_s if stim_dur is None else min(iso_s, settle_s + float(stim_dur)))
            active |= (preview_t >= t0) & (preview_t < t1)

    else:
        # Motion-series protocols: preview as pulses during active command window.
        active |= np.isfinite(preview_t)

    s1_prev = pulse * active.astype(float) * s1v
    s2_prev = pulse * active.astype(float) * s2v

stim_on_fraction = float(np.mean((s1_prev > 0) | (s2_prev > 0))) if preview_t.size > 0 else 0.0

preview_extra_df = pd.DataFrame([{
    "test_type": str(getattr(bender, "test_type", "unknown")),
    "n_samples": int(preview_t.size),
    "duration_s": float(preview_t[-1] - preview_t[0]) if preview_t.size > 1 else 0.0,
    "strain_min_pct": float(np.nanmin(strain_pct_preview)) if strain_pct_preview.size else np.nan,
    "strain_max_pct": float(np.nanmax(strain_pct_preview)) if strain_pct_preview.size else np.nan,
    "stim_on_fraction": stim_on_fraction,
    "stim_pulse_rate_hz": stim_rate,
    "s1_volts": s1v,
    "s2_volts": s2v,
}])
display(preview_extra_df)

fig_prev2 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
fig_prev2.add_trace(go.Scatter(x=preview_t, y=strain_pct_preview, mode="lines", name="strain_preview_pct", line=dict(color="magenta")), row=1, col=1)
fig_prev2.add_trace(go.Scatter(x=preview_t, y=s1_prev, mode="lines", name="S1 stim preview", line=dict(color="teal")), row=2, col=1)
fig_prev2.add_trace(go.Scatter(x=preview_t, y=s2_prev, mode="lines", name="S2 stim preview", line=dict(color="gray", dash="dot")), row=2, col=1)
_tt2 = str(getattr(bender, "test_type", "unknown"))
_strain_ylab = bender.strain_yaxis_title_pct()
if _tt2 == "calibration":
    _tt2 = str(getattr(bender, "calibration_base_test_type", "dynamic"))
if _tt2 not in ("isometric", "isovelocity"):
    _strain_ylab = "ε_geom = κ·w/2 (%)"
fig_prev2.update_yaxes(title_text=_strain_ylab, row=1, col=1)
fig_prev2.update_yaxes(title_text="Stim (V)", row=2, col=1)
fig_prev2.update_xaxes(title_text="Preview time (s)", row=2, col=1)
_prev2_title = f"Pre-run strain + stimulus preview: {getattr(bender, 'test_type', 'unknown')}"
if str(getattr(bender, "test_type", "unknown")) in ("isometric", "isovelocity"):
    _prev2_title = _prev2_title + " | " + bender.strain_geometry_plot_context()
fig_prev2.update_layout(height=650, width=1100, title_text=_prev2_title)
try:
    fig_prev2.show(renderer="notebook_connected")
except Exception:
    fig_prev2.show()
display(fig_prev2)


,test_type,n_samples,duration_s,strain_min_pct,strain_max_pct,stim_on_fraction,stim_pulse_rate_hz,s1_volts,s2_volts
0,isovelocity,4008,4.0,2.0,3.705052,0.156437,75.0,10.0,10.0


# START BENDING (Universal dispatcher run) 

This is the universal run cell. It dispatches by `bender.test_type` to the correct protocol (`dynamic`, `isometric`, `isovelocity`, `calibration`, etc.) while keeping a common data structure for export.

In [24]:
print(f"DEBUG: run_experiment dispatcher using test_type={bender.test_type!r}")
bender.run_experiment(test_type=bender.test_type)


DEBUG: run_experiment using test_type='frequency_sweep'
Data will be saved to: experiment_data_frequency_sweep_000001.h5
📏 Sonometer (Left) Calibrated: -0.98 mm
📏 Sonometer (Right) Calibrated: -0.19 mm


## Optional: save inertial calibration profile (calibration runs only)

Run this after `run_experiment` when `bender.test_type == "calibration"` to write a standalone inertial calibration H5 file that can be linked in later trials.

In [ ]:
if bender.test_type == "calibration":
    cal_outfile = bender.outputfile.replace(".h5", "_inertial_calibration.h5")
    if getattr(bender, "inertial_calibration_profile", None) is not None:
        bender.save_inertial_calibration_file(cal_outfile)
        print(f"Saved inertial calibration profile: {cal_outfile}")
    else:
        print("No inertial calibration profile estimated; check run and primary torque axis settings.")
else:
    print("Skipping inertial calibration save (test_type is not 'calibration').")


Splitting Transposed aidata ((8, 22000)) into 8 Calibrated channels...
✅ EXPORT FINISHED: Check the console output above to confirm mapping.


## Save ALL data (PRIMARY export path)

Use this exporter for all procedures. It writes schema-v2 metadata, procedure tags, and full per-trial time series from `bender.trial_records` when available.

In [ ]:
# Preferred exporter: writes one or many trials using bender.trial_records.
test_type = str(getattr(bender, "test_type", "unknown") or "unknown")
h5_schema_version = str(getattr(bender, "h5_schema_version", "2.0"))
post_trial_notes = str(getattr(bender, "post_trial_notes", "") or "")

cal_file = ""
if "inertial_calibration_file" in globals() and inertial_calibration_file:
    cal_file = str(inertial_calibration_file)
elif getattr(bender, "inertial_calibration_file", None):
    cal_file = str(bender.inertial_calibration_file)
use_cal = bool(globals().get("use_inertial_calibration", getattr(bender, "use_inertial_calibration", False)))
cal_available = bool(use_cal and cal_file and os.path.exists(cal_file))

trial_records = list(getattr(bender, "trial_records", []) or [])
if len(trial_records) == 0:
    trial_records = [{
        "test_type": test_type,
        "trial_index": 0,
        "cycle_index": 0,
        "t": np.asarray(getattr(bender, "t", np.array([]))),
        "angle_cmd": np.asarray(getattr(bender, "angle", np.array([]))),
        "anglevel_cmd": np.asarray(getattr(bender, "anglevel", np.array([]))),
        "tnorm": np.asarray(getattr(bender, "tnorm", np.array([]))),
        "S1stimcmd": np.asarray(getattr(bender, "S1stimcmd", np.array([]))),
        "S2stimcmd": np.asarray(getattr(bender, "S2stimcmd", np.array([]))),
        "aidata": np.asarray(getattr(bender, "aidata", np.array([]))),
        "angle_measured": np.asarray(getattr(bender, "angle_measured", np.array([]))),
        "forcetorque": np.asarray(getattr(bender, "forcetorque", np.array([]))),
        "forcetorque_raw": np.asarray(getattr(bender, "forcetorque_raw", np.array([]))),
        "forcetorque_corrected": np.asarray(getattr(bender, "forcetorque_corrected", np.array([]))),
        "inertial_torque_system_primary": np.asarray(getattr(bender, "inertial_torque_system_primary", np.array([]))),
        "inertial_torque_specimen_primary": np.asarray(getattr(bender, "inertial_torque_specimen_primary", np.array([]))),
        "inertial_torque_total_primary": np.asarray(getattr(bender, "inertial_torque_total_primary", np.array([]))),
        "primary_torque_raw": np.asarray(getattr(bender, "primary_torque_raw", np.array([]))),
        "primary_torque_corrected": np.asarray(getattr(bender, "primary_torque_corrected", np.array([]))),
    }]

with h5py.File(bender.outputfile, "w") as f:
    f.attrs["schema_version"] = h5_schema_version
    f.attrs["test_type"] = test_type
    f.attrs["post-trial notes"] = post_trial_notes

    g_meta = f.create_group("01_Metadata")
    g_meta.attrs["test_type"] = test_type
    g_meta.attrs["schema_version"] = h5_schema_version
    g_meta.attrs["post-trial notes"] = post_trial_notes

    g_cal_link = g_meta.create_group("calibration_link")
    g_cal_link.attrs["use_inertial_calibration"] = bool(use_cal)
    g_cal_link.attrs["calibration_file"] = cal_file
    g_cal_link.attrs["calibration_available"] = bool(cal_available)

    g_ts = f.create_group("02_TimeSeries")
    manifest_rows = []

    # Trial groups: complete time-series + condition annotations per trial.
    for i, rec in enumerate(trial_records):
        tg = g_ts.create_group(f"trial_{i:04d}")
        tg.attrs["trial_index"] = int(rec.get("trial_index", i))
        tg.attrs["cycle_index"] = int(rec.get("cycle_index", i))
        rec_tt = str(rec.get("test_type", test_type))
        tg.attrs["test_type"] = rec_tt
        manifest = {
            "trial_name": f"trial_{i:04d}",
            "trial_index": int(rec.get("trial_index", i)),
            "cycle_index": int(rec.get("cycle_index", i)),
            "test_type": rec_tt,
        }

        series_keys = [
            "t", "angle_cmd", "anglevel_cmd", "tnorm", "S1stimcmd", "S2stimcmd",
            "aidata", "angle_measured", "forcetorque", "forcetorque_raw", "forcetorque_corrected",
            "inertial_torque_system_primary", "inertial_torque_specimen_primary", "inertial_torque_total_primary",
            "primary_torque_raw", "primary_torque_corrected",
            "cycle_index_by_sample", "stim_type", "stim_state", "stim_side"
        ]

        # Build per-sample stimulation state labels.
        t_arr = np.asarray(rec.get("t", np.array([]))).reshape(-1)
        s1_arr = np.asarray(rec.get("S1stimcmd", np.array([]))).reshape(-1)
        s2_arr = np.asarray(rec.get("S2stimcmd", np.array([]))).reshape(-1)
        cyc_arr = np.asarray(rec.get("cycle_index_by_sample", np.array([]))).reshape(-1)
        n = int(t_arr.size)
        if n <= 0:
            n = int(max(s1_arr.size, s2_arr.size, cyc_arr.size, 0))
        if n > 0:
            s1 = np.zeros(n, dtype=float)
            s2 = np.zeros(n, dtype=float)
            s1[:min(n, s1_arr.size)] = s1_arr[:min(n, s1_arr.size)]
            s2[:min(n, s2_arr.size)] = s2_arr[:min(n, s2_arr.size)]
            stim_on_mask = (np.abs(s1) > 1e-12) | (np.abs(s2) > 1e-12)
            stim_enabled = bool(rec.get("is_stim", getattr(bender, "is_stim", False)))
            phase_state = np.full(n, "passive", dtype='<U8')
            activity_state = np.full(n, "passive", dtype='<U8')
            if stim_enabled:
                if cyc_arr.size == n:
                    valid_cyc = np.isfinite(cyc_arr)
                    if np.any(valid_cyc):
                        for cyc in np.unique(cyc_arr[valid_cyc]):
                            m = valid_cyc & (cyc_arr == cyc)
                            if np.any(stim_on_mask[m]):
                                phase_state[m] = "off"
                                phase_state[m & stim_on_mask] = "on"
                                activity_state[m] = "active"
                    else:
                        if np.any(stim_on_mask):
                            phase_state[:] = "off"
                            phase_state[stim_on_mask] = "on"
                            activity_state[:] = "active"
                else:
                    if np.any(stim_on_mask):
                        phase_state[:] = "off"
                        phase_state[stim_on_mask] = "on"
                        activity_state[:] = "active"
            side_state = np.full(n, "none", dtype='<U8')
            left_on = np.abs(s1) > 1e-12
            right_on = np.abs(s2) > 1e-12
            side_state[left_on & ~right_on] = "left"
            side_state[~left_on & right_on] = "right"
            side_state[left_on & right_on] = "both"
            rec["stim_type"] = np.asarray(activity_state, dtype='S8')
            rec["stim_state"] = np.asarray(phase_state, dtype='S8')
            rec["stim_side"] = np.asarray(side_state, dtype='S8')
            tg.attrs["stim_enabled"] = bool(stim_enabled)
            tg.attrs["stim_any_on"] = bool(np.any(stim_on_mask))
            manifest["stim_enabled"] = bool(stim_enabled)
            manifest["stim_any_on"] = bool(np.any(stim_on_mask))
        for key in series_keys:
            if key in rec and rec[key] is not None:
                arr = np.asarray(rec[key])
                if arr.size > 0:
                    tg.create_dataset(key, data=arr)

        # Save per-trial scalar conditions as attrs for easy filtering (e.g., freq, velocity, step).
        for k, v in rec.items():
            if k in series_keys or k in ("trial_index", "cycle_index", "test_type"):
                continue
            try:
                arr = np.asarray(v)
                if arr.ndim == 0:
                    vv = arr.item()
                    tg.attrs[str(k)] = vv
                    manifest[str(k)] = vv
                elif arr.size == 1:
                    vv = arr.reshape(-1)[0].item()
                    tg.attrs[str(k)] = vv
                    manifest[str(k)] = vv
            except Exception:
                tg.attrs[str(k)] = str(v)
                manifest[str(k)] = str(v)
        manifest_rows.append(manifest)

    # Compact index table for quick trial browsing/filtering.
    g_idx = g_meta.create_group("trial_index")
    g_idx.create_dataset("trial_names", data=np.array([f"trial_{i:04d}" for i in range(len(trial_records))], dtype='S'))
    g_idx.create_dataset("trial_index", data=np.array([int(r.get("trial_index", i)) for i, r in enumerate(manifest_rows)], dtype=np.int64))
    g_idx.create_dataset("cycle_index", data=np.array([int(r.get("cycle_index", i)) for i, r in enumerate(manifest_rows)], dtype=np.int64))
    g_idx.create_dataset("test_type", data=np.array([str(r.get("test_type", test_type)) for r in manifest_rows], dtype='S'))
    # Auto-index all scalar trial condition keys (phase, duty, freq, amp, etc.).
    reserved = {"trial_name", "trial_index", "cycle_index", "test_type"}
    all_keys = sorted({k for r in manifest_rows for k in r.keys() if k not in reserved})
    for cond_key in all_keys:
        col = [r.get(cond_key, np.nan) for r in manifest_rows]
        # Try numeric column first; fallback to string column.
        numeric_vals = []
        numeric_ok = True
        has_numeric = False
        for v in col:
            try:
                fv = float(v)
                numeric_vals.append(fv)
                if np.isfinite(fv):
                    has_numeric = True
            except Exception:
                numeric_ok = False
                break
        if numeric_ok and has_numeric:
            g_idx.create_dataset(cond_key, data=np.array(numeric_vals, dtype=float))
        else:
            svals = ["" if (v is None) else str(v) for v in col]
            if any(len(s) > 0 for s in svals):
                g_idx.create_dataset(cond_key, data=np.array(svals, dtype='S'))

    g_meta.attrs["n_trials"] = int(len(trial_records))

    # Inertial calibration profile snapshot (if available).
    prof = getattr(bender, "inertial_calibration_profile", None)
    if isinstance(prof, dict):
        g_ic = g_meta.create_group("inertial_calibration_profile")
        for k, v in prof.items():
            g_ic.attrs[str(k)] = v

    # Persist protocol metadata dictionary for grouped analysis.
    h5p = dict(getattr(bender, "h5_protocol_metadata", {}) or {})
    g_proto = g_meta.create_group("protocol_metadata")
    for k, v in h5p.items():
        try:
            arr = np.asarray(v)
            if arr.ndim > 0 and arr.size > 1:
                g_proto.create_dataset(str(k), data=arr)
            elif arr.ndim == 0:
                g_proto.attrs[str(k)] = arr.item()
            else:
                g_proto.attrs[str(k)] = str(v)
        except Exception:
            g_proto.attrs[str(k)] = str(v)

    # Persist compact scalar/vector settings from bender.__dict__ for reproducibility.
    g_settings = g_meta.create_group("bender_settings")
    skip_keys = {
        "aidata", "forcetorque", "angle", "anglevel", "tnorm", "t", "angledata",
        "S1stimcmd", "S2stimcmd", "trial_records"
    }
    for k, v in bender.__dict__.items():
        if k.startswith("_") or k in skip_keys or v is None:
            continue
        try:
            arr = np.asarray(v)
            if arr.ndim == 0:
                g_settings.attrs[str(k)] = arr.item()
            elif arr.size <= 512:
                g_settings.create_dataset(str(k), data=arr)
            else:
                g_settings.attrs[str(k)] = f"<omitted_large_array shape={arr.shape}>"
        except Exception:
            g_settings.attrs[str(k)] = str(v)

print(f"✅ EXPORT FINISHED (schema={h5_schema_version}, test_type={test_type}, n_trials={len(trial_records)})")


Voltage at start: 8.104 V
Calculated distance: 84.311 mm


✅ Plot displayed and saved as: c:\Users\jimen\Desktop\BenderData\RandomTest\2026-04-01_Polyurethane40A008.h5.png


## Universal QC plots (all procedure types)

This section plots whichever trial was run (dynamic, sweep, step, isometric, isovelocity, calibration), with a primary bending-axis torque panel and off-axis diagnostic panels.
If `use_sono` is enabled, sonomicrometry channels are measured and available for analysis/export.

For **isometric** and **isovelocity**, the figure title and angle-axis label include recruitment and a short note on geometric strain (ε = κ·w/2) vs ipsilateral fiber shortening/lengthening.

In [ ]:
# Universal plotting + save for any dispatched test type.
trial_records = list(getattr(bender, "trial_records", []) or [])
if len(trial_records) == 0:
    trial_records = [{
        "t": np.asarray(getattr(bender, "t", np.array([]))),
        "angle_cmd": np.asarray(getattr(bender, "angle", np.array([]))),
        "anglevel_cmd": np.asarray(getattr(bender, "anglevel", np.array([]))),
        "angle_measured": np.asarray(getattr(bender, "angle_measured", np.array([]))),
        "S1stimcmd": np.asarray(getattr(bender, "S1stimcmd", np.array([]))),
        "S2stimcmd": np.asarray(getattr(bender, "S2stimcmd", np.array([]))),
        "forcetorque": np.asarray(getattr(bender, "forcetorque", np.array([]))),
        "forcetorque_raw": np.asarray(getattr(bender, "forcetorque_raw", np.array([]))),
        "forcetorque_corrected": np.asarray(getattr(bender, "forcetorque_corrected", np.array([]))),
    }]

# Choose which trial to visualize.
qc_trial_index = int(globals().get("qc_trial_index", len(trial_records) - 1))
qc_trial_index = max(0, min(qc_trial_index, len(trial_records) - 1))
rec = trial_records[qc_trial_index]

t = np.asarray(rec.get("t", np.array([])))
angle_cmd = np.asarray(rec.get("angle_cmd", np.array([])))
angle_meas = np.asarray(rec.get("angle_measured", np.array([])))
S1 = np.asarray(rec.get("S1stimcmd", np.array([])))
S2 = np.asarray(rec.get("S2stimcmd", np.array([])))

# Prefer explicit raw/corrected channels; fallback to forcetorque.
ft_raw = np.asarray(rec.get("forcetorque_raw", rec.get("forcetorque", np.array([]))))
ft_corr = np.asarray(rec.get("forcetorque_corrected", np.array([])))

axis_key = str(getattr(bender, "primary_bending_axis", getattr(bender, "bending_axis_sensor", "zTorque"))).lower().strip()
axis_norm = {"x": "xTorque", "xtorque": "xTorque", "y": "yTorque", "ytorque": "yTorque", "z": "zTorque", "ztorque": "zTorque"}.get(axis_key, "zTorque")
axis_to_idx = {"xTorque": 3, "yTorque": 4, "zTorque": 5}
primary_idx = axis_to_idx.get(axis_norm, 5)
all_torque_axes = ["xTorque", "yTorque", "zTorque"]
off_axes = [ax for ax in all_torque_axes if ax != axis_norm]

def _torque_row(arr, idx):
    a = np.asarray(arr)
    if a.ndim == 2 and a.shape[0] >= 6:
        return a[idx, :]
    return np.array([])

primary_raw = _torque_row(ft_raw, primary_idx)
primary_corr = _torque_row(ft_corr, primary_idx)
off1 = _torque_row(ft_raw, axis_to_idx[off_axes[0]]) if len(off_axes) > 0 else np.array([])
off2 = _torque_row(ft_raw, axis_to_idx[off_axes[1]]) if len(off_axes) > 1 else np.array([])

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03)

if t.size > 0 and angle_cmd.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=angle_cmd, mode="lines", name="angle_cmd", line=dict(dash="dash", color="black")), row=1, col=1)
if t.size > 0 and angle_meas.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=angle_meas, mode="lines", name="angle_measured", line=dict(color="royalblue")), row=1, col=1)

if t.size > 0 and primary_raw.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=primary_raw, mode="lines", name=f"{axis_norm} raw", line=dict(color="firebrick")), row=2, col=1)
if t.size > 0 and primary_corr.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=primary_corr, mode="lines", name=f"{axis_norm} corrected", line=dict(color="seagreen")), row=2, col=1)

if t.size > 0 and off1.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=off1, mode="lines", name=f"{off_axes[0]} raw", line=dict(color="darkorange")), row=3, col=1)
if t.size > 0 and off2.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=off2, mode="lines", name=f"{off_axes[1]} raw", line=dict(color="purple")), row=4, col=1)

if t.size > 0 and S1.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=S1, mode="lines", name="S1 stim", line=dict(color="teal")), row=5, col=1)
if t.size > 0 and S2.size == t.size:
    fig.add_trace(go.Scatter(x=t, y=S2, mode="lines", name="S2 stim", line=dict(color="gray", dash="dot")), row=5, col=1)

proc = str(getattr(bender, "test_type", "unknown"))
_angle_title = "Angle (deg)"
if proc in ("isometric", "isovelocity"):
    _parts = bender.strain_yaxis_title_pct().split(" — ", 1)
    _angle_title = "Angle (deg) — " + (_parts[1] if len(_parts) > 1 else _parts[0])
fig.update_yaxes(title_text=_angle_title, row=1, col=1)
fig.update_yaxes(title_text=f"{axis_norm} (N-m)", row=2, col=1)
fig.update_yaxes(title_text=f"{off_axes[0]} (N-m)", row=3, col=1)
fig.update_yaxes(title_text=f"{off_axes[1]} (N-m)", row=4, col=1)
fig.update_yaxes(title_text="Stim (V)", row=5, col=1)
fig.update_xaxes(title_text="Time (s)", row=5, col=1)
_qc_cap = f"QC: {proc} | trial {qc_trial_index}"
if proc in ("isometric", "isovelocity"):
    _qc_cap = _qc_cap + "<br><sup style='font-size:11px'>" + bender.strain_geometry_plot_context() + "</sup>"
fig.update_layout(height=1400, width=1100, title_text=_qc_cap, showlegend=True)
fig.show()

# Save figure: PNG when kaleido is available, otherwise HTML fallback.
base = str(getattr(bender, "outputfile", "bender_output.h5")).replace(".h5", f"_{proc}_trial{qc_trial_index:03d}_qc")
png_path = f"{base}.png"
html_path = f"{base}.html"
try:
    fig.write_image(png_path, engine="kaleido", scale=2)
    print(f"Saved QC figure: {png_path}")
except Exception as e:
    fig.write_html(html_path)
    print(f"PNG export unavailable ({e}); saved HTML instead: {html_path}")


## Post-trial notes (saved to H5)

Enter notes after a run (setup issues, specimen condition, alignment, etc.).
These notes are saved in the H5 as `post-trial notes`.

In [ ]:
# Edit this string after the experiment, then run this cell.
post_trial_notes = ""

# Keep on bender so exporter cells pick it up.
bender.post_trial_notes = str(post_trial_notes)

# If H5 already exists, also patch notes directly into file metadata.
outfile = getattr(bender, "outputfile", None)
if outfile and os.path.exists(outfile):
    with h5py.File(outfile, "a") as f:
        f.attrs["post-trial notes"] = bender.post_trial_notes
        if "01_Metadata" in f:
            f["01_Metadata"].attrs["post-trial notes"] = bender.post_trial_notes
    print(f"Updated post-trial notes in: {outfile}")
else:
    print("No existing outputfile found yet; notes stored on bender and will be saved on next export.")
